In [0]:
import pandas as pd

# DataFrame containing each user's best cluster
output_with_best_clusters = pd.DataFrame(
    columns=["user_id", "best_cluster"]
)

# DataFrame containing user demographic/device information
spotify_user_demo = pd.DataFrame(
    columns=["user_id", "age", "country", "city_tier", "device_type"]
)

# Display the DataFrames structure
print("output_with_best_clusters:")
print(output_with_best_clusters)
print("\nspotify_user_demo:")
print(spotify_user_demo)


output_with_best_clusters:
Empty DataFrame
Columns: [user_id, best_cluster]
Index: []

spotify_user_demo:
Empty DataFrame
Columns: [user_id, age, country, city_tier, device_type]
Index: []


In [0]:
# ============================================================
# Cluster Profiling + Visualization (NO Persona Labels)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

# ------------------------------------------------------------
# INPUT ASSUMPTIONS
# ------------------------------------------------------------
# output_with_best_clusters:
#   columns -> user_id, best_cluster
#
# spotify_user_demo:
#   columns -> user_id, age, country, city_tier, device_type, subscription_tenure_months
# ------------------------------------------------------------

# ------------------------------------------------------------
# 1) Join clusters with demographics
# ------------------------------------------------------------

try:
    persona_df = output_with_best_clusters.merge(
        spotify_user_demo,
        on="user_id",
        how="left"
    )
except NameError as e:
    missing_var = str(e).split("'")[1]
    raise NameError(
        f"Required DataFrame '{missing_var}' is not defined. "
        f"Please ensure the following DataFrames are created before running this cell:\n"
        f"  - output_with_best_clusters: DataFrame with columns [user_id, best_cluster]\n"
        f"  - spotify_user_demo: DataFrame with columns [user_id, age, country, city_tier, device_type, subscription_tenure_months]"
    ) from None

print("Final joined shape:", persona_df.shape)
if len(persona_df) > 0:
    display(persona_df.head())
else:
    print("DataFrame is empty - no data to display")

# ------------------------------------------------------------
# 2) Cluster size & share distribution
# ------------------------------------------------------------

cluster_sizes = persona_df["best_cluster"].value_counts().sort_index()
cluster_share = (cluster_sizes / cluster_sizes.sum()).round(3)

cluster_dist = pd.DataFrame({
    "cluster_size": cluster_sizes,
    "cluster_share": cluster_share
})

print("\nCluster Size & Share:")
if len(cluster_dist) > 0:
    display(cluster_dist)
else:
    print("No cluster data available")

# ------------------------------------------------------------
# 3) Numeric Profiling
# ------------------------------------------------------------

# Check which numeric columns are actually available
available_numeric_cols = []
if "age" in persona_df.columns:
    available_numeric_cols.append("age")
if "subscription_tenure_months" in persona_df.columns:
    available_numeric_cols.append("subscription_tenure_months")

if len(available_numeric_cols) > 0 and len(persona_df) > 0:
    numeric_profile = (
        persona_df
        .groupby("best_cluster")[available_numeric_cols]
        .mean()
        .round(2)
    )
else:
    numeric_profile = pd.DataFrame()

print("\nNumeric Means by Cluster:")
if len(numeric_profile) > 0:
    display(numeric_profile)
else:
    print("No numeric profile data available")

# Index vs overall population
if len(available_numeric_cols) > 0 and len(persona_df) > 0:
    overall_means = persona_df[available_numeric_cols].mean()
    numeric_index = (numeric_profile / overall_means).round(3)
else:
    numeric_index = pd.DataFrame()

print("\nNumeric Index vs Overall (cluster_mean / overall_mean):")
if len(numeric_index) > 0:
    display(numeric_index)
else:
    print("No numeric index data available")

# ------------------------------------------------------------
# 4) Categorical Profiling — Proportional Distributions
# ------------------------------------------------------------

country_dist = pd.crosstab(
    persona_df["best_cluster"],
    persona_df["country"],
    normalize="index"
).round(3)

print("\nCountry Distribution by Cluster:")
if len(country_dist) > 0:
    display(country_dist)
else:
    print("No country distribution data available")

city_tier_dist = pd.crosstab(
    persona_df["best_cluster"],
    persona_df["city_tier"],
    normalize="index"
).round(3)

print("\nCity Tier Distribution by Cluster:")
if len(city_tier_dist) > 0:
    display(city_tier_dist)
else:
    print("No city tier distribution data available")

device_dist = pd.crosstab(
    persona_df["best_cluster"],
    persona_df["device_type"],
    normalize="index"
).round(3)

print("\nDevice Type Distribution by Cluster:")
if len(device_dist) > 0:
    display(device_dist)
else:
    print("No device distribution data available")

# ------------------------------------------------------------
# 5) Visualizations
# ------------------------------------------------------------

if len(persona_df) > 0:
    # ---- Cluster Size Distribution
    plt.figure(figsize=(6,4))
    persona_df["best_cluster"].value_counts().sort_index().plot(
        kind="bar", edgecolor="black"
    )
    plt.title("Cluster Size Distribution")
    plt.xlabel("Cluster")
    plt.ylabel("Users")
    plt.tight_layout()
    plt.show()

    # ---- Age Distribution by Cluster
    if "age" in persona_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(
            data=persona_df,
            x="best_cluster",
            y="age"
        )
        plt.title("Age Distribution by Cluster")
        plt.xlabel("Cluster")
        plt.ylabel("Age")
        plt.tight_layout()
        plt.show()

    # ---- Tenure Distribution by Cluster (only if column exists)
    if "subscription_tenure_months" in persona_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(
            data=persona_df,
            x="best_cluster",
            y="subscription_tenure_months"
        )
        plt.title("Subscription Tenure Distribution by Cluster")
        plt.xlabel("Cluster")
        plt.ylabel("Tenure (Months)")
        plt.tight_layout()
        plt.show()

    # ---- Device Distribution (Stacked Bar)
    if len(device_dist) > 0:
        device_dist.plot(
            kind="bar",
            stacked=True,
            figsize=(8,5),
            edgecolor="black"
        )
        plt.title("Device Type Distribution by Cluster")
        plt.xlabel("Cluster")
        plt.ylabel("Proportion")
        plt.legend(title="Device", bbox_to_anchor=(1.05,1), loc="upper left")
        plt.tight_layout()
        plt.show()

    # ---- Country Distribution (Top 6 countries only for readability)
    if len(persona_df["country"].value_counts()) > 0:
        top_countries = persona_df["country"].value_counts().head(6).index

        country_dist_top = pd.crosstab(
            persona_df["best_cluster"],
            persona_df["country"].where(persona_df["country"].isin(top_countries), "Other"),
            normalize="index"
        ).round(3)

        if len(country_dist_top) > 0:
            country_dist_top.plot(
                kind="bar",
                stacked=True,
                figsize=(9,5),
                edgecolor="black"
            )

            plt.title("Country Distribution by Cluster (Top Countries)")
            plt.xlabel("Cluster")
            plt.ylabel("Proportion")
            plt.legend(title="Country", bbox_to_anchor=(1.05,1), loc="upper left")
            plt.tight_layout()
            plt.show()
else:
    print("\nNo data available for visualizations")

# ------------------------------------------------------------
# 6) Heatmap — Numeric Persona Profile
# ------------------------------------------------------------

if len(numeric_profile) > 0:
    plt.figure(figsize=(7,4))
    sns.heatmap(
        numeric_profile.T,
        annot=True,
        fmt=".1f",
        cmap="coolwarm",
        linewidths=0.5
    )
    plt.title("Numeric Profile Heatmap")
    plt.xlabel("Cluster")
    plt.ylabel("Metrics")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric profile data available for heatmap")

Final joined shape: (0, 6)
DataFrame is empty - no data to display

Cluster Size & Share:
No cluster data available

Numeric Means by Cluster:
No numeric profile data available

Numeric Index vs Overall (cluster_mean / overall_mean):
No numeric index data available

Country Distribution by Cluster:
No country distribution data available

City Tier Distribution by Cluster:
No city tier distribution data available

Device Type Distribution by Cluster:
No device distribution data available

No data available for visualizations
No numeric profile data available for heatmap
